# Notebook 1 - Data Loading + Conversion & EDA


In [ ]:
#  !pip install polars pyarrow rapidfuzz pycountry statsmodels tqdm

In [ ]:
!pip uninstall polars -y
!pip install --no-cache-dir polars

In [ ]:
import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.interpolate import UnivariateSpline
from rapidfuzz import process, fuzz
import pycountry
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
FOLDER_PATH  = Path(r"path/to/your/data") 
PARQUET_DIR  = Path(r"path/to/your/data")
OUTPUT_PATH  = Path(r"path/to/your/data")

PARQUET_DIR.mkdir(exist_ok=True)
OUTPUT_PATH.mkdir(exist_ok=True)

#transaction date
TXN_DATE_COL  = "TXN_MON_DT"   
#travel date     
DEP_DATE_COL  = "TRIP_OND_DT"  
#passenger count     
PAX_COL       = "sum_pax" 
#destination location          
DEST_CTY_COL  = "FARE_OND_ARR_CNTRY_CD"

In [ ]:
import sys
import polars as pl

print("Python:", sys.executable)
print("Polars version:", pl.__version__)
print("Polars path:", pl.__file__)

## CSV -> Parquet conversion

In [ ]:
CSV_FILES = {
    "DACHBE_DACHBE": FOLDER_PATH / "DACHBE-DACHBE.csv",
    "DACHB_EU":      FOLDER_PATH / "DACHB-EU.csv",
    "EU_DACHB":      FOLDER_PATH / "EU-DACHB.csv",
    "EU_EU":         FOLDER_PATH / "EU-EU.csv",
}

COMBINED_PARQUET = PARQUET_DIR / "lhg_combined.parquet"

if not COMBINED_PARQUET.exists():
    print("Converting CSVs to Parquet")
    frames = []
    for name, csv_path in CSV_FILES.items():
        print(f"  Scanning {name}...")
        lf = (
            pl.scan_csv(
                csv_path,
                try_parse_dates=True,
                null_values=["", "NA", "N/A", "null"],
                infer_schema_length=50_000,
            )
            .filter(pl.col(PAX_COL) > 0)        # drop zero-pax rows immediately
            .with_columns(pl.lit(name).alias("segment"))
        )
        frames.append(lf)

    #Concatenate all 4 lazy frames and write in one streaming pass
    combined_lf = pl.concat(frames, how="diagonal")
    combined_lf.sink_parquet(
        COMBINED_PARQUET,
        compression="zstd",
    )
    print(f"  Done → {COMBINED_PARQUET}")
else:
    print(f"Parquet already exists at {COMBINED_PARQUET} - skipping conversion.")

#Now will always read from parquet
bookings = pl.read_parquet(COMBINED_PARQUET)
print(f"\nLoaded: {bookings.shape[0]:,} rows × {bookings.shape[1]} columns")
print(f"Memory : {bookings.estimated_size('mb'):.1f} MB in RAM")

In [ ]:
print(bookings.schema)
bookings.head(3)

In [ ]:
print("TXN date range:", bookings[TXN_DATE_COL].min(), "→", bookings[TXN_DATE_COL].max())
print("DEP date range:", bookings[DEP_DATE_COL].min(), "→", bookings[DEP_DATE_COL].max())

## Feature engineering

In [ ]:
bookings = bookings.with_columns([
    #Lead time: days between booking and travel
    (pl.col(DEP_DATE_COL) - pl.col(TXN_DATE_COL)).dt.total_days().alias("lead_time_days"),

    #Month-level periods
    pl.col(TXN_DATE_COL).dt.truncate("1mo").alias("booking_month"),
    pl.col(DEP_DATE_COL).dt.truncate("1mo").alias("travel_month"),

    #Day of week
    pl.col(TXN_DATE_COL).dt.weekday().alias("dow_txn"),
])

print("Lead time distribution (days):")
print(bookings["lead_time_days"].describe())

In [ ]:
# Lead time histogram
lt = bookings.filter(
    (pl.col("lead_time_days") >= 0) & (pl.col("lead_time_days") <= 365)
)["lead_time_days"].to_numpy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(lt, bins=100, edgecolor='none', alpha=0.75)
ax.axvline(np.median(lt), color='crimson', linestyle='--', label=f'Median {np.median(lt):.0f}d')
ax.set_title("Booking lead time distribution (days before departure)")
ax.set_xlabel("Lead time (days)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "lead_time_dist.png", dpi=150)
plt.show()

## Booking vs Travel demand

In [ ]:
travel_demand = (
    bookings
    .group_by("travel_month")
    .agg(pl.col(PAX_COL).sum().alias("pax"))
    .sort("travel_month")
    .to_pandas()
)

booking_demand = (
    bookings
    .group_by("booking_month")
    .agg(pl.col(PAX_COL).sum().alias("pax"))
    .sort("booking_month")
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(travel_demand["travel_month"].astype(str),
        travel_demand["pax"], label="Travel Demand", linewidth=2)
ax.plot(booking_demand["booking_month"].astype(str),
        booking_demand["pax"], label="Booking Activity", linewidth=2, linestyle='--')
ax.set_xticks(ax.get_xticks()[::2])
plt.xticks(rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.legend()
ax.set_title("Booking Activity vs Actual Travel Demand")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "booking_vs_travel.png", dpi=150)
plt.show()

## Deseasonalization

In [ ]:
daily_pd = (
    bookings
    .group_by(TXN_DATE_COL)
    .agg(pl.col(PAX_COL).sum().alias("pax"))
    .sort(TXN_DATE_COL)
    .to_pandas()
    .set_index(TXN_DATE_COL)
    .squeeze()
    .asfreq("D")
    .fillna(0))

stl = STL(daily_pd, period=7, robust=True)
stl_result = stl.fit()
deseasonalized = daily_pd - stl_result.seasonal

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
axes[0].plot(daily_pd, linewidth=0.7, alpha=0.8); axes[0].set_title("Original")
axes[1].plot(stl_result.seasonal, linewidth=0.7, color='orange'); axes[1].set_title("Seasonal component (weekly)")
axes[2].plot(stl_result.trend, linewidth=1.5, color='green'); axes[2].set_title("Trend component")
axes[3].plot(stl_result.resid, linewidth=0.5, color='red', alpha=0.7); axes[3].set_title("Residual")
for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
plt.suptitle("STL Decomposition — Daily Bookings (period=7)", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "stl_decomposition.png", dpi=150)
plt.show()

In [ ]:
x = np.arange(len(deseasonalized))
y = deseasonalized.values

#Moving average
ma_trend = deseasonalized.rolling(window=12, center=True).mean()

#LOWESS
lowess_result = lowess(y, x, frac=0.08)
lowess_series = pd.Series(lowess_result[:, 1], index=deseasonalized.index)

#Quadratic polynomial
coef = np.polyfit(x, y, 2)
poly_trend = pd.Series(np.polyval(coef, x), index=deseasonalized.index)

#HP Filter
cycle, hp_trend = hpfilter(deseasonalized, lamb=1600)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(deseasonalized, label='Deseasonalized', alpha=0.25, linewidth=1.5, color='gray')
ax.plot(ma_trend,      label='Moving Average (12w)', linewidth=2)
ax.plot(lowess_series, label='LOWESS', linewidth=2)
ax.plot(poly_trend,    label='Quadratic Polynomial', linewidth=2, linestyle='--')
ax.plot(hp_trend,      label='HP Filter', linewidth=2, linestyle=':')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
ax.legend()
ax.set_title('Trend comparison after removing weekly seasonality')
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (deseasonalized)')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "trend_comparison.png", dpi=150)
plt.show()

In [ ]:
# Spike detection on the deseasonalized series
# These spikes are what we want Reddit to predict

roll_mean = deseasonalized.rolling(28, center=True, min_periods=7).mean()
roll_std  = deseasonalized.rolling(28, center=True, min_periods=7).std()
z_score   = (deseasonalized - roll_mean) / roll_std

SPIKE_THRESHOLD = 2.0   # |z| > 2 = anomalous demand
spikes = z_score[z_score.abs() > SPIKE_THRESHOLD]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(deseasonalized, linewidth=0.8, alpha=0.7, label='Deseasonalized demand')
axes[0].scatter(spikes.index, deseasonalized[spikes.index],
                color='crimson', s=20, zorder=5, label=f'Spikes (|z|>{SPIKE_THRESHOLD})')
axes[0].legend()
axes[0].set_title('Deseasonalized booking demand with anomalies flagged')

axes[1].plot(z_score, linewidth=0.8, color='steelblue', label='z-score')
axes[1].axhline( SPIKE_THRESHOLD, color='crimson', linestyle='--', linewidth=1)
axes[1].axhline(-SPIKE_THRESHOLD, color='crimson', linestyle='--', linewidth=1)
axes[1].set_title(f'Rolling z-score (28-day window) — threshold ±{SPIKE_THRESHOLD}')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "booking_spikes.png", dpi=150)
plt.show()

print(f"\nTotal spike days detected: {len(spikes)}")
print("\nTop positive spikes (unexpected demand surges):")
print(z_score.nlargest(10))
print("\nTop negative spikes (unexpected demand drops):")
print(z_score.nsmallest(10))

## Destination level daily booking panels

In [ ]:
# Aggregate to country × transaction date
bookings_daily = (
    bookings
    .group_by([TXN_DATE_COL, DEST_CTY_COL])
    .agg(pl.col(PAX_COL).sum().alias("pax"))
    .rename({TXN_DATE_COL: "date", DEST_CTY_COL: "lhg_country"})
    .sort(["lhg_country", "date"])
)

print(f"Booking daily panel: {bookings_daily.shape}")
print(f"Countries: {bookings_daily['lhg_country'].n_unique()}")
bookings_daily.head(5)

## Load Reddit & destination matching

In [ ]:
reddit = pl.read_csv(
    FOLDER_PATH / "reddit.csv",
    try_parse_dates=True,
    null_values=["", "NA"],
)

if reddit["day"].dtype == pl.Utf8:
    reddit = reddit.with_columns(
        pl.col("day").str.strptime(pl.Date, "%d/%m/%Y", strict=False).alias("day")
    )

print(f"Reddit: {reddit.shape}")
print(f"Destinations: {reddit['destination'].n_unique()} unique")
reddit.head(3)

In [ ]:
reddit_dests  = reddit["destination"].unique().sort().to_list()
lhg_countries = bookings_daily["lhg_country"].unique().drop_nulls().sort().to_list()

In [ ]:

MANUAL_OVERRIDES = {
    # "UK":      "GB",
    "Turkey": "TR",
    # "USA":     "US",
    # "Czechia": "CZ",
}

FUZZY_THRESHOLD = 95

def alpha2_to_name(code):
    """Convert alpha-2 code to full country name via pycountry."""
    try:
        return pycountry.countries.get(alpha_2=code).name
    except Exception:
        return None

def alpha2_to_alpha3(code):
    """Convert alpha-2 code to alpha-3 via pycountry."""
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except Exception:
        return None


lhg_name_to_code  = {}
lhg_alpha3_to_code = {}

for code in lhg_countries:
    name = alpha2_to_name(code)
    if name:
        lhg_name_to_code[name.lower()] = code
    a3 = alpha2_to_alpha3(code)
    if a3:
        lhg_alpha3_to_code[a3] = code

lhg_full_names   = list(lhg_name_to_code.keys())
lhg_name_display = {v.lower(): alpha2_to_name(v) for v in lhg_countries if alpha2_to_name(v)}

def reddit_dest_to_alpha3(name):
    """Try to resolve a Reddit destination string to alpha-3."""
    try:
        return pycountry.countries.search_fuzzy(name)[0].alpha_3
    except Exception:
        return None

def match_destination(dest):
    # 1. Manual override
    if dest in MANUAL_OVERRIDES:
        return dict(reddit_dest=dest, lhg_country=MANUAL_OVERRIDES[dest], method="manual")

    norm = dest.strip().lower()

    # 2. Exact match on full country name (e.g. "Austria" → "AT")
    if norm in lhg_name_to_code:
        return dict(reddit_dest=dest, lhg_country=lhg_name_to_code[norm], method="exact")

    # 3. pycountry bridge: Reddit name → alpha-3 → alpha-2
    #    handles variants like "Republic of Albania", "Deutschland", etc.
    a3 = reddit_dest_to_alpha3(dest)
    if a3 and a3 in lhg_alpha3_to_code:
        return dict(reddit_dest=dest, lhg_country=lhg_alpha3_to_code[a3], method="pycountry")

    # 4. Fuzzy match against full country names (not codes)
    hit = process.extractOne(
        norm,
        lhg_full_names,
        scorer=fuzz.WRatio,
        score_cutoff=FUZZY_THRESHOLD
    )
    if hit:
        code = lhg_name_to_code[hit[0]]
        return dict(reddit_dest=dest, lhg_country=code, method=f"fuzzy({hit[1]:.0f})")

    # 5. No match
    return dict(reddit_dest=dest, lhg_country=None, method="unmatched")


match_results = [match_destination(d) for d in tqdm(reddit_dests, desc="Matching")]
match_df = pd.DataFrame(match_results)

match_df["lhg_country_name"] = match_df["lhg_country"].map(
    lambda c: alpha2_to_name(c) if pd.notna(c) else None
)

print("\n── Match breakdown ──")
print(match_df["method"].str.split("(").str[0].value_counts())

print("\n── Review fuzzy matches (check these carefully) ──")
print(match_df[match_df.method.str.startswith("fuzzy")]
      [["reddit_dest", "lhg_country", "lhg_country_name", "method"]]
      .to_string(index=False))

print("\n── Unmatched (non-European destinations — expected) ──")
print(match_df[match_df.method == "unmatched"]["reddit_dest"].tolist())

In [ ]:
#Print every alpha-2 code in LHG with its full name
print("All LHG destination countries:")
for code in sorted(lhg_countries):
    print(f"  {code} → {alpha2_to_name(code)}")

In [ ]:
match_pl = pl.from_pandas(match_df[["reddit_dest","lhg_country"]].dropna())

reddit_matched = (
    reddit
    .join(match_pl, left_on="destination", right_on="reddit_dest", how="left")
)

reddit_daily = (
    reddit_matched
    .filter(pl.col("lhg_country").is_not_null())
    .group_by(["day", "lhg_country"])
    .agg([
        pl.col("engagement").sum().alias("total_engagement"),
        pl.col("engagement_score").sum().alias("total_engagement_score"),
        pl.col("weighted_sentiment").sum().alias("total_weighted_sentiment"),
        (pl.col("weighted_sentiment").sum() /
         pl.col("engagement_score").sum()).alias("mean_sentiment"),
        pl.len().alias("n_posts"),
    ])
    .rename({"day": "date"})
    .sort(["lhg_country", "date"])
)

match_pct = reddit_matched["lhg_country"].drop_nulls().len() / len(reddit_matched) * 100
print(f"Reddit rows matched to LHG country: {match_pct:.1f}%")
print(f"Reddit daily panel: {reddit_daily.shape}")
reddit_daily.head(5)

## Save outputs

In [ ]:
bookings_daily.write_parquet(OUTPUT_PATH / "bookings_daily.parquet")
reddit_daily.write_parquet(OUTPUT_PATH / "reddit_daily.parquet")
match_df.to_csv(OUTPUT_PATH / "destination_match_table.csv", index=False)

deseas_df = pd.DataFrame({
    "date": deseasonalized.index,
    "pax_original": daily_pd.values,
    "pax_deseasonalized": deseasonalized.values,
    "trend_stl": stl_result.trend,
    "z_score": z_score.values,
})
pl.from_pandas(deseas_df).write_parquet(OUTPUT_PATH / "bookings_deseasonalized.parquet")

print("Saved:")
print("  outputs/bookings_daily.parquet")
print("  outputs/reddit_daily.parquet")
print("  outputs/bookings_deseasonalized.parquet")
print("  outputs/destination_match_table.csv")

In [ ]:
print(bookings_daily.shape)
print(reddit_daily.shape)

In [ ]:
reddit_daily = pl.read_parquet(OUTPUT_PATH / "reddit_daily.parquet")

# Should be 0
duplicates = reddit_daily.filter(
    pl.struct(["date", "lhg_country"]).is_duplicated()
)
print(f"Duplicate (date, country) pairs: {len(duplicates)}")
print(reddit_daily.head(8))